In [75]:
import chromadb
import pandas as pd

client = chromadb.PersistentClient(path=r"d:\NLP\dhl_chromadb")
collection = client.get_collection("dhl_intelligence")

print("Documents:", collection.count())

Documents: 302


In [76]:
# =========================================================
# STAGE 1 FIX: Score-based strategic_analysis (no order bias)
# =========================================================
#
# FIX NOTE: this cell previously contained TWO functions merged into one
# broken block \u2014 strategic_analysis_dashboard() ended with a `return`, but
# the tie-handling print-based logic for strategic_analysis() continued
# directly after that `return` as orphaned code outside any function. Cell
# 2 then called strategic_analysis(...), which was never actually defined
# anywhere, causing a NameError. Below, both functions are properly
# separated and closed, and both are usable independently.

import re

CATEGORY_KEYWORDS = {
    # "robot" removed as a bare substring \u2014 it was matching inside
    # "HappyRobot" (an AI agent PRODUCT NAME) and wrongly inflating the
    # automation score for AI-agent articles. "robotics"/"robotic" is kept
    # since that's unambiguous. \b regex matching avoids this class of
    # substring-collision bug entirely.
    #
    # "frame" maps each category onto the brief's Opportunity / Risk / Trend
    # dashboard buckets. A few categories (automation, ai, sustainability)
    # are framed as BOTH an Opportunity and a Trend, since the evidence
    # genuinely supports two readings at once.
    "automation": {
        "keywords": ["automation", "robotics", "robotic", "autostore", "warehouse"],
        "recommendation": "Accelerate warehouse automation and robotics deployment.",
        "impact": ["Cost Reduction", "Faster Deliveries", "Scalability"],
        "frame": ["Opportunity", "Trend"],
    },
    "energy": {
        "keywords": ["battery", "renewable", "energy", r"\bev\b", "electric"],
        "recommendation": "Increase investment in New Energy Logistics.",
        "impact": ["Revenue Growth", "Market Differentiation",
                   "Stronger Position in Renewable Energy Supply Chains"],
        "frame": ["Opportunity"],
    },
    "sustainability": {
        "keywords": ["sustainability", "gogreen", "carbon", "emission"],
        "recommendation": "Expand sustainable logistics initiatives.",
        "impact": ["Brand Reputation", "Regulatory Readiness", "ESG Positioning"],
        "frame": ["Opportunity", "Trend"],
    },
    "ecommerce": {
        "keywords": ["fulfillment", "e commerce", "ecommerce", "online retail", "parcel"],
        "recommendation": "Expand e-commerce fulfillment capabilities.",
        "impact": ["Revenue Growth", "Customer Acquisition", "Market Share"],
        "frame": ["Opportunity"],
    },
    "partnership": {
        "keywords": ["partnership", "collaboration", "alliance", "acquisition", "agreement"],
        "recommendation": "Pursue strategic partnerships to expand market reach.",
        "impact": ["Market Expansion", "Capability Access", "Risk Sharing"],
        "frame": ["Opportunity"],
    },
    "risk_tariff": {
        "keywords": ["tariff", "customs", "trade war", "regulation", "regulatory"],
        "recommendation": "Diversify supply chains to reduce tariff and regulatory exposure.",
        "impact": ["Reduced Regulatory Risk", "Business Continuity", "Supply Chain Resilience"],
        "frame": ["Risk"],
    },
    "ai": {
        # \bai\b matches standalone "AI" anywhere (start/end of string,
        # punctuation-adjacent), not just space-padded " ai ", which
        # silently missed "AI." "AI," "AI-powered" etc.
        "keywords": ["artificial intelligence", r"\bai\b", "machine learning",
                     "ai agent", "ai-powered", "ai-driven", "predictive analytics"],
        "recommendation": "Scale AI-driven operational and customer-facing capabilities.",
        "impact": ["Operational Efficiency", "Customer Experience", "Competitive Differentiation"],
        "frame": ["Opportunity", "Trend"],
    },
}

# Risk overlay \u2014 separate from the 7 categories above. Tuned 2026-06-20
# against a DIRECT corpus-wide keyword count (not semantic search): tariff
# =17 docs, customs=6, regulation+regulatory=3, layoff=2, strike=2,
# disrupt/shortage/delay/rival=1 each. competitor, lawsuit, fine, penalty,
# backlash, criticism = 0 hits and were dropped as pure noise.
RISK_SIGNAL_KEYWORDS = {
    "regulatory": ["tariff", "customs", "regulation", "regulatory", "trade war"],
    "negative_sentiment": ["layoff", "job cut", "strike", "disrupt"],
    "supply_chain": ["shortage", "delay", "bottleneck", "port congestion"],
}

MIN_SCORE_THRESHOLD = 2  # raw occurrence count across retrieved docs


def _count_occurrences(keyword: str, text: str) -> int:
    """Count ALL occurrences, not just presence \u2014 a keyword mentioned 7
    times is much stronger evidence than one mentioned once."""
    if r"\b" in keyword:
        return len(re.findall(keyword, text))
    return text.count(keyword)


def _display_label(keyword: str) -> str:
    """Strip regex syntax for clean display (e.g. '\bai\b' -> 'ai')."""
    return keyword.replace(r"\b", "").strip()


def score_categories(docs_text: str) -> dict:
    """Score every category by TOTAL keyword occurrence count. No early
    exit, no order bias, repeated strong signals outweigh incidental hits."""
    scores = {}
    for category, info in CATEGORY_KEYWORDS.items():
        counts = {kw: _count_occurrences(kw, docs_text) for kw in info["keywords"]}
        matched = {_display_label(kw): c for kw, c in counts.items() if c > 0}
        scores[category] = {
            "score": sum(matched.values()),
            "matched_keywords": list(matched.keys()),
            "keyword_counts": matched,
        }
    return scores


def _confidence_score(best_score: int, runner_up_score: int) -> tuple:
    """Confidence derived from how decisively the winner beat the runner-up."""
    if best_score == 0:
        return "low", 0.0
    ratio = best_score / (best_score + runner_up_score) if (best_score + runner_up_score) > 0 else 1.0
    if ratio >= 0.75:
        label = "high"
    elif ratio >= 0.55:
        label = "medium"
    else:
        label = "low"
    return label, round(ratio, 2)


def _severity_level(score: int) -> str:
    """Severity derived from raw evidence volume, not hardcoded per category."""
    if score >= 10:
        return "High"
    elif score >= 4:
        return "Medium"
    return "Low"


def detect_risk_signals(docs_text: str) -> dict:
    """Independent overlay: checks for risk language regardless of which
    primary category won."""
    signals = {}
    for risk_type, keywords in RISK_SIGNAL_KEYWORDS.items():
        hits = {kw: docs_text.count(kw) for kw in keywords if kw in docs_text}
        if hits:
            signals[risk_type] = {"score": sum(hits.values()), "matched_keywords": list(hits.keys())}
    return signals


In [77]:
def strategic_analysis(query, n_results=5, verbose_scores=False):
    """
    Print-based exploratory version with explicit tie-handling: when two
    or more categories score equally at the top, BOTH are reported rather
    than silently picking whichever happens to be first in the dict.
    """
    results = collection.query(query_texts=[query], n_results=n_results)
    docs = results["documents"][0]
    docs_text = " ".join(docs).lower()

    scores = score_categories(docs_text)

    best_score = max(scores[c]["score"] for c in scores)
    top_categories = [c for c in scores if scores[c]["score"] == best_score]

    if best_score >= MIN_SCORE_THRESHOLD:
        recommendations = []
        impact = []
        matched = []
        for cat in top_categories:
            info = CATEGORY_KEYWORDS[cat]
            recommendations.append(info["recommendation"])
            impact.extend(info["impact"])
            matched.extend(scores[cat]["matched_keywords"])
        impact = list(dict.fromkeys(impact))  # de-dupe, preserve order
    else:
        recommendations = ["Continue monitoring market developments."]
        impact = ["Market Awareness"]
        matched = []
        top_categories = ["general"]

    print("=" * 60)
    print("STRATEGIC RECOMMENDATION")
    print("=" * 60)
    print("\nQuery:", query)

    if len(recommendations) > 1:
        print(f"\nEvidence supports {len(recommendations)} tied categories "
              f"({', '.join(top_categories)}) \u2014 showing both rather than "
              f"arbitrarily picking one:")
        for r in recommendations:
            print("-", r)
    else:
        print("\nRecommendation:")
        print(recommendations[0])

    print("\nSupporting Evidence:")
    for i, doc in enumerate(docs, 1):
        print(f"{i}. {doc[:120]}")

    print("\nExpected Impact:")
    for item in impact:
        print("-", item)

    if matched:
        print("\nMatched signal keywords:", matched)

    if verbose_scores:
        print("\n[debug] All category scores (total occurrence count):")
        for cat, s in sorted(scores.items(), key=lambda x: -x[1]["score"]):
            print(f"  {cat}: {s['score']}  {s['keyword_counts']}")

    return {
        "query": query,
        "recommendation": recommendations[0] if len(recommendations) == 1 else recommendations,
        "categories": top_categories,
        "impact": impact,
        "evidence": docs,
        "scores": scores,
    }

In [78]:
def strategic_analysis_dashboard(query, n_results=5):
    """
    Same retrieval + scoring as strategic_analysis(), but returns a
    structured dict shaped for the Executive Intelligence Dashboard
    (brief Sections 3/4) instead of printing to stdout: Opportunity
    Monitor needs {title, impact level, evidence, confidence}. Risk
    Monitor needs {title, risk category, severity, evidence, confidence}.
    """
    results = collection.query(query_texts=[query], n_results=n_results)
    docs = results["documents"][0]
    docs_text = " ".join(docs).lower()

    scores = score_categories(docs_text)
    sorted_cats = sorted(scores.items(), key=lambda x: -x[1]["score"])
    best_cat, best_info = sorted_cats[0]
    runner_up_score = sorted_cats[1][1]["score"] if len(sorted_cats) > 1 else 0

    confidence_label, confidence_ratio = _confidence_score(best_info["score"], runner_up_score)
    severity = _severity_level(best_info["score"])
    risk_signals = detect_risk_signals(docs_text)

    entries = []
    if best_info["score"] >= MIN_SCORE_THRESHOLD:
        cat_def = CATEGORY_KEYWORDS[best_cat]
        for frame in cat_def["frame"]:
            entries.append({
                "frame": frame, "title": cat_def["recommendation"], "category": best_cat,
                "impact_level": severity, "confidence": confidence_label,
                "confidence_score": confidence_ratio, "evidence": docs,
                "matched_keywords": best_info["keyword_counts"],
            })

    for risk_type, sig in risk_signals.items():
        c_label, c_score = _confidence_score(sig["score"], 0)
        entries.append({
            "frame": "Risk", "title": f"Potential risk signal: {risk_type.replace('_', ' ')}",
            "category": risk_type, "impact_level": _severity_level(sig["score"]),
            "confidence": c_label, "confidence_score": c_score, "evidence": docs,
            "matched_keywords": sig["matched_keywords"],
        })

    return {"query": query, "entries": entries, "raw_scores": scores}

In [79]:
def strategic_analysis(query, n_results=5, verbose_scores=False):
    """
    Print-based exploratory version with explicit tie-handling: when two
    or more categories score equally at the top, BOTH are reported rather
    than silently picking whichever happens to be first in the dict.
    """
    results = collection.query(query_texts=[query], n_results=n_results)
    docs = results["documents"][0]
    docs_text = " ".join(docs).lower()

    scores = score_categories(docs_text)

    best_score = max(scores[c]["score"] for c in scores)
    top_categories = [c for c in scores if scores[c]["score"] == best_score]

    if best_score >= MIN_SCORE_THRESHOLD:
        recommendations = []
        impact = []
        matched = []
        for cat in top_categories:
            info = CATEGORY_KEYWORDS[cat]
            recommendations.append(info["recommendation"])
            impact.extend(info["impact"])
            matched.extend(scores[cat]["matched_keywords"])
        impact = list(dict.fromkeys(impact))  # de-dupe, preserve order
    else:
        recommendations = ["Continue monitoring market developments."]
        impact = ["Market Awareness"]
        matched = []
        top_categories = ["general"]

    print("=" * 60)
    print("STRATEGIC RECOMMENDATION")
    print("=" * 60)
    print("\nQuery:", query)

    if len(recommendations) > 1:
        print(f"\nEvidence supports {len(recommendations)} tied categories "
              f"({', '.join(top_categories)}) \u2014 showing both rather than "
              f"arbitrarily picking one:")
        for r in recommendations:
            print("-", r)
    else:
        print("\nRecommendation:")
        print(recommendations[0])

    print("\nSupporting Evidence:")
    for i, doc in enumerate(docs, 1):
        print(f"{i}. {doc[:120]}")

    print("\nExpected Impact:")
    for item in impact:
        print("-", item)

    if matched:
        print("\nMatched signal keywords:", matched)

    if verbose_scores:
        print("\n[debug] All category scores (total occurrence count):")
        for cat, s in sorted(scores.items(), key=lambda x: -x[1]["score"]):
            print(f"  {cat}: {s['score']}  {s['keyword_counts']}")

    return {
        "query": query,
        "recommendation": recommendations[0] if len(recommendations) == 1 else recommendations,
        "categories": top_categories,
        "impact": impact,
        "evidence": docs,
        "scores": scores,
    }

In [80]:
def strategic_analysis_dashboard(query, n_results=5):
    """
    Same retrieval + scoring as strategic_analysis(), but returns a
    structured dict shaped for the Executive Intelligence Dashboard
    (brief Sections 3/4) instead of printing to stdout: Opportunity
    Monitor needs {title, impact level, evidence, confidence}. Risk
    Monitor needs {title, risk category, severity, evidence, confidence}.
    """
    results = collection.query(query_texts=[query], n_results=n_results)
    docs = results["documents"][0]
    docs_text = " ".join(docs).lower()

    scores = score_categories(docs_text)
    sorted_cats = sorted(scores.items(), key=lambda x: -x[1]["score"])
    best_cat, best_info = sorted_cats[0]
    runner_up_score = sorted_cats[1][1]["score"] if len(sorted_cats) > 1 else 0

    confidence_label, confidence_ratio = _confidence_score(best_info["score"], runner_up_score)
    severity = _severity_level(best_info["score"])
    risk_signals = detect_risk_signals(docs_text)

    entries = []
    if best_info["score"] >= MIN_SCORE_THRESHOLD:
        cat_def = CATEGORY_KEYWORDS[best_cat]
        for frame in cat_def["frame"]:
            entries.append({
                "frame": frame, "title": cat_def["recommendation"], "category": best_cat,
                "impact_level": severity, "confidence": confidence_label,
                "confidence_score": confidence_ratio, "evidence": docs,
                "matched_keywords": best_info["keyword_counts"],
            })

    for risk_type, sig in risk_signals.items():
        c_label, c_score = _confidence_score(sig["score"], 0)
        entries.append({
            "frame": "Risk", "title": f"Potential risk signal: {risk_type.replace('_', ' ')}",
            "category": risk_type, "impact_level": _severity_level(sig["score"]),
            "confidence": c_label, "confidence_score": c_score, "evidence": docs,
            "matched_keywords": sig["matched_keywords"],
        })

    return {"query": query, "entries": entries, "raw_scores": scores}

In [81]:
strategic_analysis("DHL strategic partnerships", verbose_scores=True)
strategic_analysis("DHL sustainability strategy", verbose_scores=True)
strategic_analysis("DHL e commerce fulfillment growth", verbose_scores=True)
strategic_analysis("DHL AI analytics machine learning", verbose_scores=True)
strategic_analysis("DHL warehouse automation opportunities", verbose_scores=True)
strategic_analysis("DHL renewable energy logistics opportunities", verbose_scores=True)

STRATEGIC RECOMMENDATION

Query: DHL strategic partnerships

Recommendation:
Expand e-commerce fulfillment capabilities.

Supporting Evidence:
1. dhl ecommerce usps enter 10 billion plus long term exclusive agreement dhl ecommerce reaffirms exclusivity relationship 
2. sustainability dhl fulfillment network united states dhl fulfillment network goes beyond sustainable partner sustainabil
3. dhl supply chain erweitert die logistikpartnerschaft mit iglo august 18 2022 dhl supply chain der auf kontraktlogistik s
4. dhl supply chain dhl supply chain world leading logistics provider combining strength dhl group capabilities proven adap
5. dhl home global logistics international shipping united supply chain division creates custom solutions enterprise sized 

Expected Impact:
- Revenue Growth
- Customer Acquisition
- Market Share

Matched signal keywords: ['fulfillment', 'ecommerce']

[debug] All category scores (total occurrence count):
  ecommerce: 5  {'fulfillment': 2, 'ecommerce': 3}
  s

{'query': 'DHL renewable energy logistics opportunities',
 'recommendation': 'Increase investment in New Energy Logistics.',
 'categories': ['energy'],
 'impact': ['Revenue Growth',
  'Market Differentiation',
  'Stronger Position in Renewable Energy Supply Chains'],
 'evidence': ['dhl logistics things insights world logistics battery energy storage systems dhl new energy logistics bridging gap renewable electricity generation demand',
  'dhl targets 3bn new energy logistics revenue 2030 1 week ago dhl group expanding presence new energy sector aiming increase revenue related logistics activities approximately 600 million 2025 3 billion 2030 governments industries invest energy resilience',
  'dhl targets growing new energy logistics market air cargo news dhl investing services targeting new energy logistics market looks double revenues generated sector target sector company add new time definite service also continuing invest electric vehicle battery facilities bonn headquartered comp